# 08 - CTD distractor scaling and robust SFT

This notebook tests whether LoRA SFT can improve robustness to irrelevant biomedical evidence. It builds clean, distractor, and no-path tasks from CTD chemical-gene and curated gene-disease relations.

Two training conditions are compared:
1. Vanilla SFT: clean evidence only.
2. Distractor-aware SFT: clean examples plus distractor examples.

The evaluation sweeps the number of distractor edges and reports disease accuracy, path accuracy, and no-path accuracy. This notebook is self-contained and does not depend on another notebook's runtime state.

In [ ]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl==0.29.1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"

In [ ]:
import os, re, random, ast
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')

CHEM_GENE = '/content/CTD_chem_gene_ixns.tsv.gz'
GENE_DISEASE = '/content/CTD_curated_genes_diseases.tsv.gz'

if not os.path.exists(CHEM_GENE) or not os.path.exists(GENE_DISEASE):
    print('Upload: CTD_chem_gene_ixns.tsv.gz and CTD_curated_genes_diseases.tsv.gz')
    files.upload()

assert os.path.exists(CHEM_GENE), 'Missing CTD_chem_gene_ixns.tsv.gz'
assert os.path.exists(GENE_DISEASE), 'Missing CTD_curated_genes_diseases.tsv.gz'

print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
chem_cols = ['ChemicalName','ChemicalID','CasRN','GeneSymbol','GeneID','GeneForms','Organism','OrganismID','Interaction','InteractionActions','PubMedIDs']
gd_cols = ['GeneSymbol','GeneID','DiseaseName','DiseaseID','DirectEvidence','InferenceChemicalName','InferenceChemicalID','OmimIDs','PubMedIDs']
chem = pd.read_csv(CHEM_GENE, sep='\t', comment='#', header=None, names=chem_cols, dtype=str, low_memory=False)
gd = pd.read_csv(GENE_DISEASE, sep='\t', comment='#', header=None, names=gd_cols, dtype=str, low_memory=False)

chem = chem[chem['OrganismID'].fillna('').str.strip().eq('9606')].copy()
chem = chem[chem['ChemicalName'].notna() & chem['GeneSymbol'].notna() & chem['GeneID'].notna()].copy()
gd = gd[gd['GeneID'].notna() & gd['DiseaseName'].notna() & gd['DiseaseID'].notna()].copy()
chem['GeneID'] = chem['GeneID'].str.replace(r'\.0$', '', regex=True)
gd['GeneID'] = gd['GeneID'].str.replace(r'\.0$', '', regex=True)
gd = gd.drop_duplicates(['GeneID','DiseaseID'])

pairs = chem.merge(gd[['GeneID','DiseaseName','DiseaseID']], on='GeneID', how='inner')
pairs = pairs.drop_duplicates(['ChemicalID','GeneID','DiseaseID']).reset_index(drop=True)
pairs = pairs[['ChemicalName','ChemicalID','GeneSymbol','GeneID','DiseaseName','DiseaseID']].dropna()
print('2-hop paths:', len(pairs))
pairs.head()

In [ ]:
# Make a chemically disjoint evaluation set.
rng = random.Random(42)
unique_chems = pairs['ChemicalID'].drop_duplicates().tolist()
rng.shuffle(unique_chems)
eval_chems = set(unique_chems[:max(1, int(0.1*len(unique_chems)))])
train_pool = pairs[~pairs['ChemicalID'].isin(eval_chems)].copy()
eval_pool = pairs[pairs['ChemicalID'].isin(eval_chems)].copy()

# Keep this first experiment small enough for Colab.
train_pool = train_pool.head(4500).copy()
eval_pool = eval_pool.head(250).copy()
print('Train paths:', len(train_pool), 'Eval paths:', len(eval_pool), 'Eval chemicals:', eval_pool['ChemicalID'].nunique())

In [ ]:
def clean_prompt(row):
    return (
        f"Evidence 1: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n"
        f"Evidence 2: gene {row.GeneSymbol} is linked to disease {row.DiseaseName} in the curated CTD gene-disease data.\n"
        f"Question: What disease is connected to {row.ChemicalName} through gene {row.GeneSymbol}? "
        'Return `Disease: <name>` and `Path: Chemical -> Gene -> Disease`.'
    )

def clean_answer(row):
    return f"Disease: {row.DiseaseName}. Path: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}."


In [ ]:
# Build distractor examples for training and evaluation.
def make_distractor(row, k, rng):
    candidates = pairs[(pairs['GeneSymbol'] != row.GeneSymbol) & (pairs['DiseaseName'] != row.DiseaseName)]
    if len(candidates) < k:
        return None
    ds = candidates.sample(n=k, random_state=rng.randint(0, 10**9))
    edges = [f"{row.GeneSymbol} -> {row.DiseaseName}"] + [f"{r.GeneSymbol} -> {r.DiseaseName}" for _, r in ds.iterrows()]
    rng.shuffle(edges)
    prompt = (
        f"Evidence A: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n"
        + 'Gene-disease evidence:\n- ' + '\n- '.join(edges)
        + f"\nQuestion: Using only the evidence above, what disease is connected to {row.ChemicalName} through gene {row.GeneSymbol}? "
          'Return `Disease: <name>` and `Path: Chemical -> Gene -> Disease`.'
    )
    answer = f"Disease: {row.DiseaseName}. Path: {row.ChemicalName} -> {row.GeneSymbol} -> {row.DiseaseName}."
    return prompt, answer

def make_no_path(row, rng):
    candidates = pairs[(pairs['GeneSymbol'] != row.GeneSymbol) & (pairs['DiseaseName'] != row.DiseaseName)]
    r = candidates.sample(n=2, random_state=rng.randint(0, 10**9)).iloc[0]
    prompt = (
        f"Evidence A: {row.ChemicalName} has a CTD chemical-gene relationship with gene {row.GeneSymbol}.\n"
        f"Gene-disease evidence: {r.GeneSymbol} -> {r.DiseaseName}.\n"
        f"Question: Is there a supported Chemical -> Gene -> Disease path from {row.ChemicalName} through gene {row.GeneSymbol} in the evidence above? "
        'Answer YES or NO and give a one-sentence justification.'
    )
    return prompt, 'NO. The supplied gene-disease edge does not use the queried gene.'


In [ ]:
# Evaluation sets: clean + distractor counts + no-path.
eval_sets = {'clean': []}
for k in [1, 3, 5, 10]:
    eval_sets[f'distractor_{k}'] = []
eval_sets['no_path'] = []

for _, row in eval_pool.iterrows():
    eval_sets['clean'].append({'prompt': clean_prompt(row), 'target_disease': row.DiseaseName, 'target_gene': row.GeneSymbol, 'target_chemical': row.ChemicalName, 'kind':'clean'})
    for k in [1, 3, 5, 10]:
        item = make_distractor(row, k, rng)
        if item is not None:
            p, a = item
            eval_sets[f'distractor_{k}'].append({'prompt':p, 'target_disease':row.DiseaseName, 'target_gene':row.GeneSymbol, 'target_chemical':row.ChemicalName, 'kind':f'distractor_{k}'})
    p, a = make_no_path(row, rng)
    eval_sets['no_path'].append({'prompt':p, 'target_disease':None, 'target_gene':row.GeneSymbol, 'target_chemical':row.ChemicalName, 'kind':'no_path'})

for k,v in eval_sets.items():
    print(k, len(v))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def render(prompt, answer=None):
    msgs = [{'role':'user','content':prompt}]
    if answer is not None:
        msgs.append({'role':'assistant','content':answer})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=answer is None)

def generate(model, prompts, batch_size=8, max_new_tokens=80):
    model.eval(); outs=[]
    for s in range(0, len(prompts), batch_size):
        batch = prompts[s:s+batch_size]
        enc = tokenizer([render(p) for p in batch], return_tensors='pt', padding=True, truncation=True, max_length=384)
        enc = {k:v.to(model.device) for k,v in enc.items()}
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        n = enc['input_ids'].shape[1]
        outs.extend(tokenizer.batch_decode(out[:,n:], skip_special_tokens=True))
    return outs

In [ ]:
def score_set(items, preds):
    disease_hits=[]; path_hits=[]; no_path_hits=[]
    for item,pred in zip(items,preds):
        p = re.sub(r'[^a-z0-9]+',' ',pred.lower())
        if item['kind']=='no_path':
            ok = 'no' in p and 'yes' not in p[:20]
            no_path_hits.append(ok)
        else:
            d = re.sub(r'[^a-z0-9]+',' ',item['target_disease'].lower())
            g = re.sub(r'[^a-z0-9]+',' ',item['target_gene'].lower())
            c = re.sub(r'[^a-z0-9]+',' ',item['target_chemical'].lower())
            disease_hits.append(d in p)
            path_hits.append(d in p and g in p and c in p)
    out={}
    if disease_hits:
        out['disease_accuracy']=sum(disease_hits)/len(disease_hits)
        out['path_accuracy']=sum(path_hits)/len(path_hits)
    if no_path_hits:
        out['no_path_accuracy']=sum(no_path_hits)/len(no_path_hits)
    return out

In [ ]:
# Helper for SFT dataset creation.
def make_sft_dataset(df, distractor_prob=0.0, seed=42, n_limit=3500):
    rng_local=random.Random(seed); rows=[]
    sampled=df.sample(min(n_limit,len(df)), random_state=seed)
    for _,row in sampled.iterrows():
        if rng_local.random() < distractor_prob:
            item=make_distractor(row, 3, rng_local)
            if item is not None:
                p,a=item
            else:
                p,a=clean_prompt(row),clean_answer(row)
        else:
            p,a=clean_prompt(row),clean_answer(row)
        rows.append({'text':render(p,a)})
    return Dataset.from_list(rows)


In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def train_sft(train_ds, output_dir):
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).cuda()
    model.config.use_cache=False
    lora=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=['q_proj','k_proj','v_proj','o_proj'],bias='none',task_type='CAUSAL_LM')
    args=SFTConfig(output_dir=output_dir,per_device_train_batch_size=2,gradient_accumulation_steps=4,num_train_epochs=1,learning_rate=2e-4,max_length=512,logging_steps=25,save_strategy='no',report_to='none',gradient_checkpointing=False,packing=False,fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    trainer=SFTTrainer(model=model,args=args,train_dataset=train_ds,processing_class=tokenizer,peft_config=lora)
    trainer.train(); return model


In [ ]:
# Train two conditions.
vanilla_ds=make_sft_dataset(train_pool, distractor_prob=0.0, seed=1)
robust_ds=make_sft_dataset(train_pool, distractor_prob=0.5, seed=2)
print('Vanilla examples:',len(vanilla_ds),'Robust examples:',len(robust_ds))

print('Training vanilla SFT...')
vanilla_model=train_sft(vanilla_ds,'./outputs/08-vanilla-sft')
del vanilla_model
torch.cuda.empty_cache()

print('Training distractor-aware SFT...')
robust_model=train_sft(robust_ds,'./outputs/08-robust-sft')

In [ ]:
# Load/evaluate both models independently to avoid LoRA state confusion.
def evaluate_model(model_path, label):
    m=AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).cuda()
    from peft import PeftModel
    m=PeftModel.from_pretrained(m, model_path)
    results={}
    for name,items in eval_sets.items():
        preds=generate(m,[x['prompt'] for x in items])
        results[name]=score_set(items,preds)
    print(label,results)
    del m; torch.cuda.empty_cache()
    return results


In [ ]:
# The training outputs are adapter directories even though save_strategy='no'; save them explicitly.
vanilla_dir='./outputs/08-vanilla-adapter'
robust_dir='./outputs/08-robust-adapter'
# robust_model is still in memory. Save it.
robust_model.save_pretrained(robust_dir); tokenizer.save_pretrained(robust_dir)
del robust_model; torch.cuda.empty_cache()

# Reload and save a vanilla adapter by retraining from the same seed/config would cost time, so use the trainer checkpoint path if available.
# For a clean self-contained comparison, re-train vanilla once and immediately save the adapter.
vanilla_model=train_sft(vanilla_ds,'./outputs/08-vanilla-sft-final')
vanilla_model.save_pretrained(vanilla_dir); tokenizer.save_pretrained(vanilla_dir)
del vanilla_model; torch.cuda.empty_cache()


In [ ]:
vanilla_results=evaluate_model(vanilla_dir,'Vanilla SFT')
robust_results=evaluate_model(robust_dir,'Distractor-aware SFT')


In [ ]:
print('\nROBUSTNESS TABLE')
print('-'*92)
print(f"{'Setting':<20}{'Vanilla disease':>18}{'Robust disease':>18}{'Vanilla path':>16}{'Robust path':>16}")
print('-'*92)
for name in ['clean','distractor_1','distractor_3','distractor_5','distractor_10']:
    v=vanilla_results.get(name,{}) ; r=robust_results.get(name,{})
    print(f"{name:<20}{v.get('disease_accuracy',float('nan')):>18.3f}{r.get('disease_accuracy',float('nan')):>18.3f}{v.get('path_accuracy',float('nan')):>16.3f}{r.get('path_accuracy',float('nan')):>16.3f}")
print(f"{'no_path':<20}{vanilla_results['no_path'].get('no_path_accuracy',float('nan')):>18.3f}{robust_results['no_path'].get('no_path_accuracy',float('nan')):>18.3f}")


## Interpretation

The central hypothesis is that vanilla SFT improves clean evidence composition but does not necessarily teach robust relation selection. Distractor-aware SFT is expected to improve performance as the number of irrelevant gene-disease edges increases.

The no-path task tests whether the model can abstain when the queried gene has no supplied disease edge. These synthetic tasks are diagnostic benchmarks, not biomedical claims.

For a paper-quality experiment, repeat across multiple seeds, report confidence intervals, and add model-size and gene-disjoint/disease-disjoint splits.